In [1]:
import os
import torch
import torch_geometric.nn as nn
from torch_geometric.data import DataLoader

from models.dataset import GraphDataset
from models.base_gat_struct import GATClassifier
from models.model_utils import train, test

## GAT with dummy features per line

In [2]:
# Define Parameters
in_channels = 10
hidden_channels = 20
out_channels = 5 # EQUIVALENT OF A 1 LAYER MLP AT THE END.
num_layers = 5
dropout = 0.1
act = "relu"


In [3]:
# Initialise dataset and dataloader
dataset = GraphDataset("./json_output/", struct_thresh=0.6, textural_thresh=0.4)

loader = DataLoader(dataset, batch_size=2, shuffle=True)


/home/Soufiane/.conda/envs/3dv/lib/python3.11/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


In [4]:
# Initialize model
model = GATClassifier(in_channels=in_channels, 
                        hidden_channels=hidden_channels, 
                        out_channels=out_channels, 
                        num_layers=num_layers, 
                        dropout=dropout, 
                        act=act)

# Loss function and Optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
criterion = torch.nn.BCELoss()

In [5]:
epochs = 5

for epoch in range(1, epochs+1):
    loss = train(model, optimizer, criterion, loader)
    train_acc, val_acc, test_acc = test(model, optimizer, criterion, loader, threshold=0.5)
    print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}, Train Acc: {train_acc:.4f}, Val Acc: {val_acc:.4f}, Test Acc: {test_acc:.4f}')

Epoch: 001, Loss: 1.4049, Train Acc: 0.7482, Val Acc: 0.7469, Test Acc: 0.7495
Epoch: 002, Loss: 1.1919, Train Acc: 0.7547, Val Acc: 0.7261, Test Acc: 0.7511
Epoch: 003, Loss: 1.1136, Train Acc: 0.7695, Val Acc: 0.7783, Test Acc: 0.6546
Epoch: 004, Loss: 1.3039, Train Acc: 0.7446, Val Acc: 0.7401, Test Acc: 0.7670
Epoch: 005, Loss: 1.2726, Train Acc: 0.6724, Val Acc: 0.6460, Test Acc: 0.6396


# DEEPLSD features

In [4]:
import cv2
import torch.nn.functional as F
from deeplsd.models.deeplsd_inference import DeepLSD
from line_understanding.utility_methods import load_color_image


In [5]:
# DeepLSD

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
conf = {'detect_lines': True, 'line_detection_params': {'merge': False, 'filtering': True, 'grad_thresh': 3}}
ckpt = torch.load('../weights/deeplsd_md.tar', map_location='cpu', weights_only=False)
net = DeepLSD(conf)
net.load_state_dict(ckpt['model'])
net = net.to(device).eval()

In [ ]:
# Define hook functions to capture features

df_intermediate_features = None
angle_intermediate_features = None

def hook_df(module, input, output):
    global df_intermediate_features
    df_intermediate_features = output.detach()

def hook_angle(module, input, output):
    global angle_intermediate_features
    angle_intermediate_features = output.detach()

# Register hooks on the layer
df_hook_handle = net.df_head[5].register_forward_hook(hook_df)
angle_hook_handle = net.angle_head[5].register_forward_hook(hook_angle)


In [8]:
def sample_line_features(feature_map, line, num_samples=10, downsample_ratio=None):
    """
    Sample and concatenate the features from each sample point into a single vector.
    
    Args:
        feature_map (torch.Tensor): Combined feature map of shape (B, C, H, W).
        line (array-like): A 2x2 array where line[0] is (x1, y1) and line[1] is (x2, y2)
                           in the original image coordinate system.
        num_samples (int): Number of points sampled along the line.
        downsample_ratio (float): The factor by which the original image is downsampled.
        
    Returns:
        concatenated_feature (torch.Tensor): Concatenated embedding for the line,
                                               of shape (C * num_samples,).
    """
    # Unpack endpoints (each endpoint is [x, y])
    (x1, y1), (x2, y2) = line
    # Convert coordinates to match the feature map resolution.
    x1, y1, x2, y2 = x1 / downsample_ratio, y1 / downsample_ratio, x2 / downsample_ratio, y2 / downsample_ratio

    # Create uniformly spaced sampling points along the line using linear interpolation.
    t_vals = torch.linspace(0, 1, steps=num_samples, device=feature_map.device)
    xs = x1 + t_vals * (x2 - x1)
    ys = y1 + t_vals * (y2 - y1)

    # Build a sampling grid; grid_sample expects normalized coordinates in [-1, 1].
    grid = torch.stack([xs, ys], dim=-1)  # shape: (num_samples, 2)
    # Get spatial dimensions of the feature map (assumed shape: (B, C, H, W))
    _, _, H, W = feature_map.shape
    grid[..., 0] = (grid[..., 0] / (W - 1)) * 2 - 1  # Normalize x coordinates.
    grid[..., 1] = (grid[..., 1] / (H - 1)) * 2 - 1  # Normalize y coordinates.
    grid = grid.unsqueeze(0).unsqueeze(2)  # Reshape to (1, num_samples, 1, 2) for grid_sample

    # Use bilinear interpolation to sample the feature map.
    # This returns a tensor of shape (B, C, num_samples, 1)
    sampled_features = torch.nn.functional.grid_sample(feature_map, grid, align_corners=True)
    # Remove the extra dimension to obtain shape (B, C, num_samples)
    sampled_features = sampled_features.squeeze(-1)
    
    # Instead of aggregating via mean, we concatenate the features at the sampled points.
    # Reshape sampled_features from shape (B, C, num_samples) to (B, C * num_samples)
    concatenated_feature = sampled_features.view(sampled_features.shape[0], -1)
    
    # Remove the batch dimension if batch_size == 1, resulting in a tensor of shape (C * num_samples,).
    return concatenated_feature.squeeze(0)
    

(414, 2, 2)


In [14]:
# Process the images and extract line embeddings.
frame_str = "0001"
desired_images = ["ai_001_001"]
base_data_dir = "data"

for image_id in desired_images:
    # Reset the intermediate features
    df_intermediate_features = None
    angle_intermediate_features = None

    #Load image
    image_dir = os.path.join(base_data_dir, image_id)
    cam_view_color = "scene_cam_00_final_preview"
    
    color_img = load_color_image(image_dir, image_id, frame_str, cam_view_color)
    h, w = color_img.shape[:2]
    gray_img = cv2.cvtColor(color_img, cv2.COLOR_RGB2GRAY)

    # Run DeepLSD
    input_tensor = torch.tensor(gray_img, dtype=torch.float32, device=device)[None, None] / 255.
    with torch.no_grad():
        out = net({'image': input_tensor})
        pred_lines = out['lines'][0]
        if isinstance(pred_lines, torch.Tensor):
            pred_lines = pred_lines.cpu().numpy()
            
    
    # Stack the features from the two heads
    combined_features = torch.cat([df_intermediate_features, angle_intermediate_features], dim=1)
    downsample_ratio = color_img.shape[1] / combined_features.shape[3]
    
    # Generate embeddings for each detected line.
    line_embeddings = []
    for line in pred_lines:
        embedding = sample_line_features(combined_features, line, num_samples=10, downsample_ratio=downsample_ratio)
        line_embeddings.append(embedding.cpu().numpy())
    
    # Output results.
    print(f"Image {image_id}: Detected {len(pred_lines)} lines.")
    if line_embeddings:
        print(f"Sample embedding shape: {line_embeddings[0].shape}")
    print(line_embeddings[0])
        



data/ai_001_001/ai_001_001/images/scene_cam_00_final_preview/frame.0001.color.jpg
Image ai_001_001: Detected 414 lines.
Sample embedding shape: (1280,)
[-0.00141568 -0.00141568 -0.00141568 ... -0.42394888 -0.42394888
 -0.42394888]


In [13]:
# Remove the hooks when they are no longer needed.
df_hook_handle.remove()
angle_hook_handle.remove()


## GAT with DeepLSD features

In [6]:
# Define Parameters
in_channels = 1280
hidden_channels = 256
out_channels = 256 # EQUIVALENT OF A 1 LAYER MLP AT THE END.
num_layers = 5
dropout = 0.1
act = "relu"


In [7]:
# Initialise dataset and dataloader
dataset = GraphDataset("./json_output/", struct_thresh=0.6, textural_thresh=0.4)

loader = DataLoader(dataset, batch_size=2, shuffle=True)


In [8]:
print(len(loader))

2


In [4]:
# Initialize model
model = GATClassifier(in_channels=in_channels, 
                        hidden_channels=hidden_channels, 
                        out_channels=out_channels, 
                        num_layers=num_layers, 
                        dropout=dropout, 
                        act=act)

# Loss function and Optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
criterion = torch.nn.BCELoss()

In [5]:
epochs = 5

for epoch in range(1, epochs+1):
    loss = train(model, optimizer, criterion, loader)
    train_acc, val_acc, test_acc = test(model, optimizer, criterion, loader, threshold=0.5)
    print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}, Train Acc: {train_acc:.4f}, Val Acc: {val_acc:.4f}, Test Acc: {test_acc:.4f}')

Epoch: 001, Loss: 22.8409, Train Acc: 0.6309, Val Acc: 0.6559, Test Acc: 0.6640
Epoch: 002, Loss: 65.6735, Train Acc: 0.6667, Val Acc: 0.6382, Test Acc: 0.7511
Epoch: 003, Loss: 34.5775, Train Acc: 0.6805, Val Acc: 0.7056, Test Acc: 0.6937
Epoch: 004, Loss: 1.3964, Train Acc: 0.6797, Val Acc: 0.6706, Test Acc: 0.6801
Epoch: 005, Loss: 1.7027, Train Acc: 0.6785, Val Acc: 0.6371, Test Acc: 0.7171
